# 21 - McNemar's test

Run after all twelve training notebooks are done.

Dietterich (1998) compared five tests for classifier comparison and found McNemar's to
be the only one with acceptable Type-I error when each model is trained once and
evaluated on a single test set. That is exactly the situation here, so it is the test
used throughout.

The comparisons are split into two families, corrected separately:

- **Confirmatory**, 9 comparisons over E1-E6. Registered before any of this was run.
- **Exploratory**, 12 comparisons involving EfficientNet-B3. Added afterwards.

Pooling them would let a post-hoc addition change whether the pre-registered results
come out significant, which is not defensible. Keeping them apart is a choice and it
gets stated openly in the write-up rather than buried in a footnote.

In [1]:
import json, sys, time
from pathlib import Path

# Works whether the kernel starts in notebooks/ or at the repo root.
ROOT = Path.cwd()
while not (ROOT / "src" / "ham10000").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

DATA = ROOT / "data"
RUNS = ROOT / "runs"
print("repo:", ROOT)

repo: /workspace/ham10000-cnn-comparison


In [2]:
from ham10000.constants import EXPERIMENTS

missing = [e for e in EXPERIMENTS if not (RUNS / e / "test_predictions.csv").exists()]
print("missing:", missing if missing else "none, all twelve are in")

missing: none, all twelve are in


In [3]:
import subprocess
r = subprocess.run([sys.executable, str(ROOT / "scripts" / "statistical_tests.py")],
                   capture_output=True, text=True)
print(r.stdout or r.stderr)

Confirmatory family: 9 comparisons over E1-E6
Exploratory family:  12 comparisons involving EfficientNet-B3


[conf]  E1 vs E2   p=1  not significant (better: E1)
[conf]  E3 vs E4   p=1  not significant (better: E3)
[conf]  E5 vs E6   p=6.066e-06  significant     (better: E5)
[conf]  E1 vs E3   p=1.616e-08  significant     (better: E1)
[conf]  E1 vs E5   p=1  not significant (better: E1)
[conf]  E3 vs E5   p=6.841e-09  significant     (better: E5)
[conf]  E2 vs E4   p=2.298e-11  significant     (better: E2)
[conf]  E2 vs E6   p=8.261e-07  significant     (better: E2)
[conf]  E4 vs E6   p=0.3676  not significant (better: E6)
[expl]  E7 vs E1   p=1  not significant (better: E7)
[expl]  E7 vs E2   p=1  not significant (better: E7)
[expl]  E8 vs E3   p=0.2217  not significant (better: E8)
[expl]  E8 vs E4   p=0.06506  not significant (better: E8)
[expl]  E9 vs E5   p=1  not significant (better: E9)
[expl]  E9 vs E6   p=1.999e-07  significant     (better: E9)
[expl]  E7 vs E8   p=1.879e-05 

In [4]:
import pandas as pd
res = pd.read_csv(RUNS / "mcnemar_results.csv")
pd.set_option("display.width", 200)
for fam in res.family.unique():
    print(f"\n===== {fam} =====")
    print(res[res.family == fam][
        ["exp_a", "exp_b", "a_only_correct", "b_only_correct",
         "p_holm_corrected", "reject_at_0.05", "better"]
    ].to_string(index=False))


===== confirmatory =====
exp_a exp_b  a_only_correct  b_only_correct  p_holm_corrected  reject_at_0.05 better
   E1    E2              82              78      1.000000e+00           False     E1
   E3    E4             114             102      1.000000e+00           False     E3
   E5    E6             143              71      6.066289e-06            True     E5
   E1    E3             162              70      1.616146e-08            True     E1
   E1    E5              84              78      1.000000e+00           False     E1
   E3    E5              53             139      6.840792e-09            True     E5
   E2    E4             150              50      2.297798e-11            True     E2
   E2    E6             133              59      8.260623e-07            True     E2
   E4    E6              97             123      3.675688e-01           False     E6

===== exploratory =====
exp_a exp_b  a_only_correct  b_only_correct  p_holm_corrected  reject_at_0.05 better
   E7    E1   

## Reading this

`a_only_correct` and `b_only_correct` are the discordant counts: images one model got
right and the other did not. McNemar's only looks at those, which is the point. Images
both models handle identically carry no information about which is better.

A non-significant result is not a null result here. It says the two configurations
disagree about equally often in both directions, which for a comparison between a 9 MB
model and a 94 MB one is itself a finding.

In [5]:
sig = res[res["reject_at_0.05"]]
print(f"{len(sig)} of {len(res)} comparisons significant after Holm correction\n")
for fam in res.family.unique():
    sub = res[res.family == fam]
    print(f"{fam:<14} {sub['reject_at_0.05'].sum()}/{len(sub)}")

9 of 21 comparisons significant after Holm correction

confirmatory   5/9
exploratory    4/12
